# Only for fact_work we need SCD 2 or dated snapshot for rest of the things we dont need SCD2 we can simply overwrite as shifts timing or anything else does not get repeated

In [0]:
# Quick Schema Check: Print exact Bronze column names
fact_tables = [
    "fact_shift", 
    "fact_work", 
    "fact_shrinkage"
]

for tbl in fact_tables:
    print(f"=== Columns in 7trents_data_platform.bronze.{tbl} ===")
    spark.table(f"7trents_data_platform.bronze.{tbl}").printSchema()

In [0]:
# OBJECTIVE: Clean fact tables and construct Daily Cumulative Snapshots
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.window import Window

bronze_schema = "7trents_data_platform.bronze"
silver_schema = "7trents_data_platform.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(f'bronze_schema = {bronze_schema}')
print(f'silver_schema = {silver_schema}')

In [0]:
# Helper to calculate duration in decimal hours
def add_duration_hours(df, start_col, end_col, target_col):
    return df.withColumn(
        target_col,
        F.round((F.col(end_col).cast("long") - F.col(start_col).cast("long")) / 3600.0, 2)
    )

In [0]:
# ------------------------------------------------------------------------------
# 1. PROCESS FACT_SHIFT (Clean & Append by Batch)
# ------------------------------------------------------------------------------
print("\n--- Processing: silver.fact_shift ---")
df_bronze_shift = spark.table(f"{bronze_schema}.fact_shift")

df_silver_shift = (
    df_bronze_shift
    .filter(F.col("shift_id").isNotNull() & F.col("people_id").isNotNull())
    .withColumn("shift_id", F.col("shift_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("scheduled_start", F.col("scheduled_start").cast(TimestampType()))
    .withColumn("scheduled_end", F.col("scheduled_end").cast(TimestampType()))
    .withColumn("shift_date", F.to_date(F.col("scheduled_start")))
    .transform(lambda df: add_duration_hours(df, "scheduled_start", "scheduled_end", "shift_hours"))
    .dropDuplicates(["shift_id", "_batch_date"])
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "shift_id",
        "people_id",
        "county_id",
        "shift_date",
        "scheduled_start",
        "scheduled_end",
        "shift_hours",
        "_source_file",
        "_batch_date",
        "_load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_shift.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.fact_shift")
)
print(f"✓ Saved {silver_schema}.fact_shift: {df_silver_shift.count()} rows")

In [0]:

# ------------------------------------------------------------------------------
# 2. PROCESS FACT_SHRINKAGE (Clean & Append by Batch)
# ------------------------------------------------------------------------------
print("\n--- Processing: silver.fact_shrinkage ---")
df_bronze_shrinkage = spark.table(f"{bronze_schema}.fact_shrinkage")

df_silver_shrinkage = (
    df_bronze_shrinkage
    .filter(F.col("shrinkage_id").isNotNull() & F.col("people_id").isNotNull())
    .withColumn("shrinkage_id", F.col("shrinkage_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("shrinkage_type_id", F.col("shrinkage_type_id").cast(IntegerType()))
    .withColumn("start_datetime", F.col("start_datetime").cast(TimestampType()))
    .withColumn("end_datetime", F.col("end_datetime").cast(TimestampType()))
    .withColumn("shrinkage_date", F.to_date(F.col("start_datetime")))
    .transform(lambda df: add_duration_hours(df, "start_datetime", "end_datetime", "shrinkage_hours"))
    .dropDuplicates(["shrinkage_id", "_batch_date"])
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "shrinkage_id",
        "people_id",
        "shrinkage_type_id",
        "shrinkage_date",
        "start_datetime",
        "end_datetime",
        "shrinkage_hours",
        "_source_file",
        "_batch_date",
        "_load_type",
        "_silver_processed_at"
    )
)

(
    df_silver_shrinkage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.fact_shrinkage")
)
print(f"✓ Saved {silver_schema}.fact_shrinkage: {df_silver_shrinkage.count()} rows")

# we can use SCD2 here also but it will be difficult to calculate plan adherence in power BI 

===============================================================

# here dated snapshot method is used because this is good for power BI point of view so that we can get - what happens in each day easily - how jobs are scheduled and completed and we can calculate the plan_adherance percentage from that

In [0]:

# ------------------------------------------------------------------------------
# 3. PROCESS FACT_WORK AS DATED SNAPSHOTS (silver.fact_work_snapshot)
# ------------------------------------------------------------------------------
print("\n--- Processing: silver.fact_work_snapshot ---")
df_bronze_work = spark.table(f"{bronze_schema}.fact_work")

# Clean, cast, and compute planned duration
df_clean_work = (
    df_bronze_work
    .filter(F.col("work_id").isNotNull())
    .withColumn("work_id", F.col("work_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("cancellation_id", F.col("cancellation_id").cast(IntegerType()))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("scheduled_start", F.col("scheduled_start").cast(TimestampType()))
    .withColumn("scheduled_end", F.col("scheduled_end").cast(TimestampType()))
    .withColumn("work_date", F.to_date(F.col("scheduled_start")))
    .transform(lambda df: add_duration_hours(df, "scheduled_start", "scheduled_end", "planned_hours"))
)

# Identify all distinct batch dates in chronological order
batch_dates = [
    row["_batch_date"] for row in 
    df_clean_work.select("_batch_date").distinct().orderBy("_batch_date").collect()
]

print(f"Constructing cumulative daily snapshots for batch dates: {batch_dates}")

cumulative_snapshots = []

# For each batch date, take all records accumulated up to that date,
# picking the latest record per work_id as of that snapshot date
for snap_date in batch_dates:
    window_spec = Window.partitionBy("work_id").orderBy(F.col("_ingested_at").desc(), F.col("_batch_date").desc())
    
    df_snap = (
        df_clean_work
        .filter(F.col("_batch_date") <= F.lit(snap_date))
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .withColumn("snapshot_date", F.lit(snap_date).cast(DateType()))
    )
    cumulative_snapshots.append(df_snap)

# Union all daily snapshots together
df_all_snapshots = cumulative_snapshots[0]
for next_snap in cumulative_snapshots[1:]:
    df_all_snapshots = df_all_snapshots.unionByName(next_snap)

df_all_snapshots = (
    df_all_snapshots
    .withColumn("_silver_processed_at", F.current_timestamp())
    .select(
        "snapshot_date",
        "work_id",
        "people_id",
        "county_id",
        "work_date",
        "status",
        "cancellation_id",
        "scheduled_start",
        "scheduled_end",
        "planned_hours",
        "_source_file",
        "_batch_date",
        "_load_type",
        "_silver_processed_at"
    )
)

(
    df_all_snapshots.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.fact_work_snapshot")
)
print(f"✓ Saved {silver_schema}.fact_work_snapshot: {df_all_snapshots.count()} rows")


In [0]:

# Validation summary
print("\nSnapshot row counts by snapshot_date and status:")
spark.table(f"{silver_schema}.fact_work_snapshot").groupBy("snapshot_date", "status").count().orderBy("snapshot_date", "status").show()